# Day 20 — Pandas merging & joins
Objectives:
- Inner, left, right, outer joins.
- Handling key collisions and suffixes.
- Efficient merges and memory tips.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-20`. Read
`python/ds-60day/companion-guides/day20_pandas_merging_joins.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A merge combines rows whose key values match. Before code, state each
table's grain, which columns form the key, and whether that key is unique
on each side. Those facts determine expected cardinality: one-to-one,
one-to-many, many-to-one, or many-to-many.

Join type controls preservation: inner keeps matches, left preserves
left rows, right preserves right rows, and outer preserves both sides.
A join can silently drop unmatched rows or multiply repeated matches.
Use `validate` to enforce cardinality, `indicator=True` to classify
matches/orphans, and row-count/total reconciliation to prove the result
has the intended meaning.

### Vocabulary

- **key:** one or more columns used to identify or match records.
- **cardinality:** the one/many relationship of matching keys on each side.
- **join:** an operation combining records according to key matches.
- **orphan:** a row whose key has no match on the other side.
- **anti-join:** rows from one side that have no match.
- **reconciliation:** checks proving expected rows and measures were preserved.

## Syntax anatomy

`left.merge(right, on="sku", how="left", validate="many_to_one",
indicator=True)` preserves all left rows, matches equal `sku` values,
asserts that the right key is unique, and adds `_merge` evidence. The
validation wording is from left to right.

### Worked example 1 — Enrich many line items from one product row

Make many-to-one cardinality executable. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import pandas as pd

items = pd.DataFrame({"sku": ["A", "A", "B"], "qty": [1, 2, 1]})
products = pd.DataFrame({"sku": ["A", "B"], "price": [10.0, 5.0]})
priced = items.merge(
    products, on="sku", how="left",
    validate="many_to_one", indicator=True
)
(len(priced), priced["_merge"].value_counts().to_dict())

**Expected observation:** `(3, {'both': 3, ...})`; category counts may include zero-valued labels. No item row was lost or multiplied.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Make an orphan visible

A left merge indicator supports reconciliation and anti-joins. Predict first; then run the next cell.

In [ ]:
more_items = pd.DataFrame({"sku": ["A", "C"], "qty": [1, 1]})
checked = more_items.merge(
    products, on="sku", how="left",
    validate="many_to_one", indicator=True
)
checked[["sku", "_merge"]].to_dict("records")

**Expected observation:** `[{'sku': 'A', '_merge': 'both'}, {'sku': 'C', '_merge': 'left_only'}]`. Product `C` is an orphan.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Check key dtypes, missingness, and uniqueness on both tables before merging.
2. Use the cardinality language from left to right when choosing `validate`.
3. Add `indicator=True` and inspect every category before dropping `_merge`.
4. Reconcile row counts and additive totals; a successful call is not proof of a correct relationship.

**Alternative to compare:** An index join can be concise when indexes are deliberate keys; explicit column merges are often easier for beginners to audit.

**Boundary to test:** Null keys, whitespace/case differences, composite keys, duplicate dimensions, and many-to-many multiplication need policy.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import pandas as pd
customers = pd.DataFrame({"cid":[1,2,3], "name":["Ada","Alan","Linus"]})
orders = pd.DataFrame({"oid":[10,11,12], "cid":[1,1,4], "amount":[100,200,300]})
customers, orders

pd.merge(customers, orders, on='cid', how='inner')
pd.merge(customers, orders, on='cid', how='left', suffixes=('_c','_o'))
pd.merge(customers, orders, on='cid', how='outer')


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Join product, order-item, and customer tables to compute revenue per customer. **Before code:** write each table's row grain and key uniqueness. **Sequence:** many order items to one product, calculate line revenue, aggregate to customer grain, then attach one customer record.
   **Expected behavior:** one output row per customer with orders.
   **Verify:** use `validate` on both merges and reconcile total line revenue to customer revenue.

2. Demonstrate a right join that preserves every row of a chosen right-side customer table, including a customer with no matching order.
   **Expected behavior:** the unmatched right row survives with missing order fields. **Then:** swap table order and reproduce the result with a left join.
   **Verify:** compare sorted keys and explain why left joins are often easier to read from a chosen primary table.

3. Create data where a supposedly unique dimension key is duplicated, then use the correct `validate` relationship to raise `MergeError`. **Constraints:** state which side should be one and which may be many; do not de-duplicate merely to silence the error.
   **Verify:** repair the fixture or data contract and show the validated merge succeeds without row multiplication.

### Additional mastery practice

Declare each table's grain and key cardinality before merging. Use validation and reconciliation to make row loss or multiplication visible.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

4. **Prediction:** One key appears twice on the left and three times on the right. Predict the number of joined rows for that key.
   **Progressive hint:** A many-to-many match forms every pair: left count × right count.
   **Verify:** Build the 2-by-3 fixture and assert exactly six rows for that key; compare with `validate` rejecting the unintended many-to-many relationship.
5. **Tracing:** Trace an outer merge with `indicator=True` and classify `left_only`, `right_only`, and `both` rows.
   **Progressive hint:** The indicator is a compact reconciliation tool.
   **Verify:** Assert one known key lands in each indicator category and reconcile category counts to the full outer-join row count.
6. **Implementation:** Implement an anti-join returning left rows whose key has no right match.
   **Progressive hint:** Use a left merge with indicator, then filter `left_only`.
   **Verify:** Assert the anti-join returns exactly the unmatched left keys, preserves left columns/order, and does not duplicate rows when right keys repeat.
7. **Debugging:** Repair a merge whose `validate='one_to_many'` is reversed relative to the actual product-to-order-item relationship.
   **Progressive hint:** Say which side must have unique keys before choosing `1:m` or `m:1`.
   **Verify:** Assert key uniqueness on each side, choose `many_to_one` for item-to-product data, and show the reversed validation fails on the duplicate item key.
8. **Edge case and explanation:** Investigate how missing keys match in pandas and decide whether to reject, sentinel-fill, or separate them before a business-key join.
   **Progressive hint:** Do not assume pandas null-key behavior matches SQL.
   **Verify:** Test two missing keys under pandas behavior, then assert the chosen reject/separate/sentinel policy prevents them from being mistaken for a business match.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Join product, order-item, and customer tables to compute revenue per customer. **Before code:** write each table's row grain and key uniqueness. **Sequence:** many order items to one product, calculate line revenue, aggregate to customer grain, then attach one customer record. **Expected behavior:** one output row per customer with orders. **Verify:** use `validate` on both merges and reconcile total line revenue to customer revenue.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Join product, order-item, and customer tables to compute revenue per customer. write each table's row grain and key uniqueness. many order items to one product, calculate line r...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Demonstrate a right join that preserves every row of a chosen right-side customer table, including a customer with no matching order. **Expected behavior:** the unmatched right row survives with missing order fields. **Then:** swap table order and reproduce the result with a left join. **Verify:** compare sorted keys and explain why left joins are often easier to read from a chosen primary table.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Demonstrate a right join that preserves every row of a chosen right-side customer table, including a customer with no matching order. the unmatched right row survives with missi...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** Create data where a supposedly unique dimension key is duplicated, then use the correct `validate` relationship to raise `MergeError`. **Constraints:** state which side should be one and which may be many; do not de-duplicate merely to silence the error. **Verify:** repair the fixture or data contract and show the validated merge succeeds without row multiplication.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Create data where a supposedly unique dimension key is duplicated, then use the correct `validate` relationship to raise `MergeError`. state which side should be one and which m...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** One key appears twice on the left and three times on the right. Predict the number of joined rows for that key. **Progressive hint:** A many-to-many match forms every pair: left count × right count. **Verify:** Build the 2-by-3 fixture and assert exactly six rows for that key; compare with `validate` rejecting the unintended many-to-many relationship.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: One key appears twice on the left and three times on the right. Predict the number of joined rows for that key. A many-to-many match forms every pair: left count × right count....
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace an outer merge with `indicator=True` and classify `left_only`, `right_only`, and `both` rows. **Progressive hint:** The indicator is a compact reconciliation tool. **Verify:** Assert one known key lands in each indicator category and reconcile category counts to the full outer-join row count.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Trace an outer merge with `indicator=True` and classify `left_only`, `right_only`, and `both` rows. The indicator is a compact reconciliation tool. Assert one known key lands in...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement an anti-join returning left rows whose key has no right match. **Progressive hint:** Use a left merge with indicator, then filter `left_only`. **Verify:** Assert the anti-join returns exactly the unmatched left keys, preserves left columns/order, and does not duplicate rows when right keys repeat.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Implement an anti-join returning left rows whose key has no right match. Use a left merge with indicator, then filter `left_only`. Assert the anti-join returns exactly the unmat...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a merge whose `validate='one_to_many'` is reversed relative to the actual product-to-order-item relationship. **Progressive hint:** Say which side must have unique keys before choosing `1:m` or `m:1`. **Verify:** Assert key uniqueness on each side, choose `many_to_one` for item-to-product data, and show the reversed validation fails on the duplicate item key.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Repair a merge whose `validate='one_to_many'` is reversed relative to the actual product-to-order-item relationship. Say which side must have unique keys before choosing `1:m` o...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 8 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Investigate how missing keys match in pandas and decide whether to reject, sentinel-fill, or separate them before a business-key join. **Progressive hint:** Do not assume pandas null-key behavior matches SQL. **Verify:** Test two missing keys under pandas behavior, then assert the chosen reject/separate/sentinel policy prevents them from being mistaken for a business match.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 8 — your work
# Short contract: Investigate how missing keys match in pandas and decide whether to reject, sentinel-fill, or separate them before a business-key join. Do not assume pandas null-key behavior mat...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
